# Latent Ablation Study: Turning Off Latent Layers

In this notebook, we take a trained NVAE model and turn off some latent layers during reconstruction and generation, to see how it deteriorates the reconstruction and generation quality.

This is a more rigorous analysis of posterior collapse. First, we can take a look at the KL divergence train graphs for each layer. If the KL divergence is close to 0, it is an indicator of potential posterior collapse. To verify this, we can turn off this layer and see if it affects evaluation performance.

## Preliminaries

The repository assumes all entry points to be run as modules from root. Since
this notebook is 2 levels deep, the below block moves us to the root directory.

In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

Update `model_path` to choose the model.

In [ ]:
import lightning as L
import torch

from arch.nvae.nvae import NVAE
from utils.const import SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import setup_device

model_path = "logs/nvae_acdc/best/default/checkpoints/epoch=97-step=20972.ckpt"

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=20)

# Reseed after preprocessing data
L.seed_everything(SEED)

# Load model
model = NVAE.load_from_checkpoint(model_path)

## Reconstruction

First, let's look at reconstruction quality.

NVAE has a shared encoder-decoder architecture. During the top-down pass, the encoder shares information with the decoder to build the residual distribution at each layer. Here, by turning off a layer, we prevent information from being shared.

In [ ]:
from utils.utils import get_data

loader_test = data_module.test_dataloader()
x = get_data(loader_test)
x.shape

Refer to the docstring of the forward pass in the NVAE decoder class (`arch/nvae/decoder.py`). Update `num_shared_layers` on line 11.

In [ ]:
from monai.losses.dice import DiceLoss

from utils.const import ACDC
from utils.eval import get_samples_and_reconstructions
from utils.utils import discretise, show_samples

with torch.no_grad():
    model.eval()
    model.to(device)
    x = x.to(device)
    
    x_hat_logits, _, _, _, _ = model(x, test=True, num_shared_layers=3)

recon_loss = model.reconstruction_loss(x, x_hat_logits)
print(f"Reconstruction loss: {recon_loss.item()}")

# Compute Dice score
x_hat = torch.softmax(x_hat_logits, dim=1)
x_hat_onehot = discretise(x_hat)
dl = DiceLoss(reduction="mean", include_background=False)
dice_score = 1 - dl(input=x_hat_onehot, target=x)
print(f"Dice score: {dice_score}")

# Ignore background class
for component_idx in range(1, ACDC.NUM_CLASSES):
    x_component = x[:, component_idx, :, :].unsqueeze(1)
    x_hat_onehot_component = x_hat_onehot[:, component_idx, :, :].unsqueeze(1)
    
    x_component = x_component.to(device)
    x_hat_onehot_component = x_hat_onehot_component.to(device)
    
    dl = DiceLoss(reduction="mean")
    dice_score = 1 - dl(input=x_hat_onehot_component, target=x_component)
    print(f"Dice score for component {ACDC.mask_classes[component_idx]}: {dice_score}")

x = x.cpu()
x_hat_logits = x_hat_logits.cpu()

samples_and_reconstructions = get_samples_and_reconstructions(x[:20], x_hat_logits[:20])
show_samples(samples_and_reconstructions, rgb=False, ncol=10, figsize=(10, 4))

## Generation

Now, let's look at generation quality.

Here, by turning off a layer, we do not sample from its corresponding Normal distribution, but instead take the mean as the latent variable.

Example: Say we set `num_sample_layers = 1`. This means the only variation comes from the topmost latent layer. If we observe the generated samples to be very similar, it means the topmost layer does not encode much information and has collapsed.

In [ ]:
with torch.no_grad():
    model.eval()
    model.to(device)

    # Generate probabilistic segmentation maps
    z_shape = model.decoder.get_top_latent_shape(batch_size=40)
    z = torch.randn(z_shape, device=device)
    
    for i in range(3):
        x_fake = model.decoder.generate(
            num_samples=40,
            device=device,
            num_sample_layers=i + 1,
            z=z,
        )
                
        feats_fake = model.conditional_coder(x_fake)

        # Discretise probabilistic map then view generations
        generations = torch.argmax(feats_fake, dim=1).unsqueeze(1)
        show_samples(generations, rgb=False, ncol=10, figsize=(10, 4))